pydantic library

In [9]:
from pydantic import BaseModel, Field

class User(BaseModel):
    name: str = Field(min_length=2)
    age: int = Field(ge=0)  # ge=0 means greater than or equal to 0
    email: str

# 1. Parsing and validating raw JSON/dict data:
raw_input = {"name": "Alice", "age": "25", "email": "alice@example.com"}
user = User(**raw_input)

print(user.age)        # 25 (automatically converted from string to int)
print(type(user.age))  # <class 'int'>

# 2. Exporting back to dict or JSON:
user_dict = user.model_dump()       # {'name': 'Alice', 'age': 25, 'email': 'alice@example.com'}
user_json = user.model_dump_json()  # '{"name":"Alice","age":25,"email":"alice@example.com"}'

25
<class 'int'>


In [ ]:
"""Generic field normalization from structured paper evidence to landscape features."""

from __future__ import annotations

import re
from typing import Callable, Iterable, Sequence

from src.extraction.evidence import EvidenceItem, PaperEvidence, canonical_evidence_key
from src.models.landscape import PaperFeatures
from src.models.paper import Paper

_VAGUE_VALUES = {
    "",
    "other",
    "unknown",
    "unspecified",
    "none",
    "not specified",
    "not reported",
}

_GENERIC_METHOD_VALUES = {
    "method",
    "methods",
    "model",
    "models",
    "approach",
    "approaches",
    "technique",
    "techniques",
    "algorithm",
    "algorithms",
    "architecture",
    "architectures",
    "framework",
    "frameworks",
    "proposed method",
    "proposed model",
    "our method",
    "our model",
}

_GENERIC_DATASET_PATTERN = re.compile(
    r"^(?:"
    r"dataset|datasets|data|benchmark|benchmark datasets?|"
    r"public datasets?|private datasets?|custom datasets?|"
    r"proprietary datasets?|several datasets?|multiple datasets?|"
    r"various datasets?|different datasets?|several public datasets?|"
    r"multiple public datasets?|widely used datasets?"
    r")$",
    re.I,
)

_SENTENCE_LIKE_PATTERN = re.compile(
    r"\b(?:we|this paper|this study|our work|the authors|"
    r"propose|proposes|proposed|develop|developed|evaluate|evaluated|"
    r"investigate|investigated|achieve|achieved|outperform|outperformed|"
    r"using|used to)\b",
    re.I,
)

def _clean(value : str) -> str :
    return canonical_evidence_key(value)

def _is_concrete_phrase(value : str, *, max_words : int = 12) -> bool :
    normalized = _clean(value)
    if not normalized or normalized in _VAGUE_VALUES :
        return False
    
    words = normalized.split()

    if not 1 <= len(words) <= max_words :
        return False
    
    return any(character.isalpha() for character in normalized)

_REVIEW_PATTERN = re.compile(
    r"\b(?:systematic review|scoping review|literature review|review|"
    r"survey|meta analysis)\b",
    re.I,
)

def normalize_problem(value : str) -> str :
    normalized = _clean(value)
    if not normalized : return ""
    return normalized if _is_concrete_phrase(normalized) else ""


def normalize_method(value : str) -> str :
    normalized = _clean(value)
    if not normalized or normalized in _GENERIC_METHOD_VALUES :
        return ""
    
    normalized = re.sub()
    return normalized if normalized not in _GENERIC_METHOD_VALUES else ""

def normalize_method_family(value : str) -> str :
    return normalize_method(value)

_DATASET_SUFFIX_PATTERN = re.compile(
    r"\s+(?:dataset|datasets|corpus|cohort)$",
    re.I,
)

def normalize_dataset(value : str) -> str :
    normalized = _clean(value)
    if not normalized : 
        return ""
    
    if _GENERIC_DATASET_PATTERN.fullmatch(normalized):
        return ""

    normalized = _DATASET_SUFFIX_PATTERN.sub("", normalized).strip()

    if not normalized or _GENERIC_DATASET_PATTERN.fullmatch(normalized):
        return ""

    return normalized if _is_concrete_phrase(normalized, max_words=12) else ""


def normalize_metric(value : str) -> str :
    normalized = _clean(value)
    return normalized if _is_concrete_phrase(normalized, max_words = 6) else ""

def metric_kind(value : str) -> str | None :
    normalized = normalize_metric(value)
    if not normalized :
        return None
    
    return "performance"


def normalize_constraint(value : str) -> str :
    normalized = _clean(value)
    if not normalized or normalized in _VAGUE_VALUES :
        return ""
    
    return normalized if _is_concrete_phrase(normalized, max_words = 12) else ""


def dataset_types(record: PaperEvidence) -> list[str]:
    """Return no inferred categories; dataset characteristics need structured evidence."""

    return []


